## 🔰PyTorchでニューラルネットワーク基礎　#37 【DataCollatorWithPadding】

### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください


### データについて
* history_text_label_id.jsonl: wikipediaの日本の歴史分野の内容から収集した時代区分を判定するテキス
ト
    * ids列の長さがそれぞれことなります。
    * 時代区分を直接表すテキストは除いてある。（江戸時代ラベルの文章に「江戸」という単語がはありま
せん）
    * クラス数：５（弥生、奈良、室町、江戸、昭和）

### TransformerEncoderを利用したBERTタイプの文章分類のネットワークの構造について

* 事前学習はseq_len=64で学習済み
* 最大の系列長が64となる

### 今回扱う内容
1. BERETタイプモデルのファインチューニング
2. DataLoaderのcollate_fnにDataCollatorWithPaddingを利用
3. tokenizerにPreTrainedTokenizerFastを利用
4. step単位で学習するために、データ生成の関数を作成

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
# from tokenizers import Tokenizer  # 今回は使わない
from transformers import PreTrainedTokenizerFast


# カスタマイズする部分
from torch.utils.data import Dataset, DataLoader

# HuggingFaceのtransformersライブラリ
from transformers import DataCollatorWithPadding

from sklearn.model_selection import train_test_split

In [2]:
#デバイスの選択
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("利用デバイス:", device)

# 精度を計算する関数
def accuracy(y, t):
    _, argmax_list = torch.max(y, dim=1)
    accuracy = sum(argmax_list == t).item()/len(t)
    return accuracy

利用デバイス: cuda:0


### tokenizerの読み込み方法が変わる
* HuggingFaceの**PreTrainedTokenizerFast**を利用する
* 注意点：special token、語彙数の呼び出し方が変わるので、いくつか変更する部分がある


In [3]:
pre_trained_model = "./model/unigram_2k.model"                # 事前学習されたモデル
data_filename = "./data/history_text_label_id.jsonl"          # 分類問題のデータ
tokenizer_filename = "tokenizer/unigram_tokenizer_2k.json"    # 保存したトークナイザーで確認

df = pd.read_json(data_filename, lines=True)

## データの系列長がそれぞれ異なる
* バッチ学習時に\<pad\>して揃えるcollate_fn関数やDatasetクラスをカスタマイズしていく

In [4]:
df.sample(3)

,text,label,ids
380,乱と卑弥呼 魏志倭人伝には、卑弥呼が倭国を治める以前は、諸国が対立し互いに攻め合っていたとい...,1,"[1, 83, 12, 3, 1745, 1941, 42, 3, 869, 1819, 6..."
164,東久邇内閣は民主化の進展に対応できず総辞職し、歴代内閣の中で最短政権を記録している。,0,"[1, 118, 1069, 3, 106, 873, 11, 776, 303, 227,..."
59,東京では、有田八郎外相とロバート・クレイギー英大使との会談が開かれた。,0,"[1, 118, 146, 298, 413, 199, 1033, 601, 325, 5..."


In [5]:
# idsの要素数がそれぞれ異なることを確認してみた
[len(df.iloc[i]["ids"]) for i in range(10)]

[12, 22, 13, 64, 14, 55, 26, 38, 34, 64]

In [6]:
# 訓練データと検証データに分割
train_data, test_data = train_test_split(df, stratify=df["label"], random_state=55)

### HFのDataCollatorWithPaddingを使う
* DataCollatorWithPaddingを使うために、変数名を指定されたものへ変更する必要がある。
* DataCollatorWithPaddingで等長化を行う。

**train_dataのidsとlabelに対して行う処理**

* カスタムDatasetクラス：train_dataのidsとlabelを辞書形式で出力させたい
* HFが要求するキーの名称：input_idsとlabels

**必要な作業**
* Datasetクラスの定義
* collate_fnの定義

### tokenizerの修正
* DataCollatorWithPaddingはtokenizerを引数に持つ。tokenizerを利用して\<pad\>挿入を行うためだ。
* 要求される変数名へコピーする
* PreTrainedTokenizerFast を使いtokenizerを修正
* SpecialTokensMixin というクラスで定義されている要求される変数名に変更

In [7]:
class SimpleDataset(Dataset):
    def __init__(self, data):
        # データフレームの列データをリストへ変換。
        # __getitem__(index)でリストのindexを利用してアクセスできるようにしてみた
        self.input_ids = data["ids"].tolist()
        self.labels = data["label"].tolist()
   
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # キーの名称はHFが設定しているものを利用する
        # input_ids, labels, attention_mask
        return {
            "input_ids": self.input_ids[idx],
            "labels": self.labels[idx]
        }

In [8]:
dataset = SimpleDataset(train_data)
print(dataset[0])   # __getitem__(index)で値が求まる

{'input_ids': [1, 1417, 45, 5, 929, 878, 143, 152, 116, 5, 324, 699, 39, 1288, 1762, 202, 45, 669, 7, 929, 93, 8, 776, 7, 1648, 121, 189, 5, 162, 462, 715, 137, 18, 6, 2], 'labels': 2}


In [9]:
# SpecialTokensMixin というクラスで定義されている「決まった名前」
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file= tokenizer_filename,
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
    mask_token="<mask>",
)

print("vocab_size:", tokenizer.vocab_size)
print("len(tokenizer):", len(tokenizer))
print("mask_token_id:", tokenizer.mask_token_id)
print("pad_token_id:", tokenizer.pad_token_id)


vocab_size: 2000
len(tokenizer): 2000
mask_token_id: 4
pad_token_id: 0


In [10]:
hf_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding= "longest",)       # longest: バッチ内で最長
    #pad_to_multiple_of=8)      # 8の倍数 つけたほうが良いらしいが、今回だとほとんど64に揃ってしまうので可変長か判別しにくいので使わない。

In [11]:
train_loader = DataLoader(
    dataset,
    batch_size=8,
    collate_fn = hf_collator,
    shuffle=True,
    drop_last=True
)

In [12]:
# 実際に学習されるバッチごとに系列長が異なることを確認
print([x["input_ids"].shape[1] for x in train_loader][:16])

[64, 64, 60, 64, 60, 64, 64, 64, 64, 58, 64, 64, 59, 53, 64, 62]


**モデルの設定**
* 事前学習時に使っていたmlm_head部分を分類問題用Linearに修正
* 文頭\<bos\>の特徴量を分類問題用Linearへ入力できるように修正

In [13]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = len(tokenizer) # 変更箇所(1) tokenizer.get_vocab_size()
        self.seq_len = 64
        self.d_model = 64
        self.nhead = 4
        self.dim_feedforward = 256
        self.num_layers = 6
        self.dropout = 0.1
        self.out_features = 5
        
        # 特殊トークンID  変更箇所(2)
        self.pad_token_id = tokenizer.pad_token_id #tokenizer.token_to_id("<pad>")
        self.mask_token_id = tokenizer.mask_token_id #tokenizer.token_to_id("<mask>")
        self.bos_token_id = tokenizer.bos_token_id # tokenizer.token_to_id("<bos>")
        self.eos_token_id = tokenizer.eos_token_id # tokenizer.token_to_id("<eos>")
        self.unk_token_id = tokenizer.unk_token_id # tokenizer.token_to_id("<unk>")

        # 特殊トークンのセット
        self.special_tokens = {
            self.pad_token_id,
            self.mask_token_id,
            self.bos_token_id,
            self.eos_token_id,
            self.unk_token_id,
        }
        
        # 通常トークンのリスト special_tokenを除くトークンのリスト（MLMランダム置換用）
        self.normal_tokens = [
            i for i in range(self.vocab_size) 
            if i not in self.special_tokens
        ]
        
        # 学習設定 (今回は利用しないけど使うと便利かも)
        self.batch_size = 512
        self.learning_rate = 0.0001
        self.num_epochs = 100
        self.mask_prob = 0.15
        self.max_grad_norm = 1.0


class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        # 埋め込み層
        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size, 
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id
        )
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers, enable_nested_tensor=False)
        
        # MLM用の出力層：BERT風レイヤー　各トークン位置で語彙全体を予測
        self.classifier = nn.Linear(in_features=config.d_model, out_features=config.out_features)

    
    def forward(self, x):
        # マスクの作成
        src_key_padding_mask = (x == self.config.pad_token_id)
        
        # 埋め込み
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(torch.arange(x.size(1), device=x.device))
        x = tok_emb + pos_emb.unsqueeze(0)
        
        x = self.layer_norm(x)
        x = self.dropout(x)
        
        # Transformer Encoder
        h = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)

        # 文ベクトルへの Pooling  <BOS>トークン（先頭）に情報を集約
        pooled = h[:, 0, :]  # [batch, d_model]
        
        # 分類 
        y = self.classifier(pooled)  # [batch, num_labels=5]    
        return y

## 学習済みモデルの読み込み

In [14]:
# 事前学習モデルの設定を分類問題用へ更新
checkpoint = torch.load(pre_trained_model, map_location=device)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])

model = DNN(config).to(device)

In [15]:
# 事前学習した重みを分類問題用へコピー
# 事前学習重み（checkpoint側）をフィルタ
pretrained = checkpoint["model_state_dict"]
pretrained_filtered = {
    k: v for k, v in pretrained.items()
    if (not k.startswith("mlm_head."))      # and (not k.startswith("classifier."))
}

# 実際にコピーする部分
# strict=Falseにすることで、ファイルにはないパラメータは読み込まれず、初期状態のまま保持される
# Missing Keys: 重み付けされないパラメータ
info = model.load_state_dict(pretrained_filtered, strict=False)

# 空のリストが表示されるならコピー成功！
print(f"削除されているか確認: {info.unexpected_keys}")

削除されているか確認: []


In [16]:
params_to_update = []

for name , param in model.named_parameters():
    param.requires_grad = False
for name, param in model.transformer_encoder.layers[-2:].named_parameters():  # 最終層＋１
    param.requires_grad = True
    params_to_update.append(param)
    print("更新されるパラメータ:", name)
for name, param in model.classifier.named_parameters():
    param.requires_grad = True
    params_to_update.append(param)
    print("更新されるパラメータ:", name) 

更新されるパラメータ: 0.self_attn.in_proj_weight
更新されるパラメータ: 0.self_attn.in_proj_bias
更新されるパラメータ: 0.self_attn.out_proj.weight
更新されるパラメータ: 0.self_attn.out_proj.bias
更新されるパラメータ: 0.linear1.weight
更新されるパラメータ: 0.linear1.bias
更新されるパラメータ: 0.linear2.weight
更新されるパラメータ: 0.linear2.bias
更新されるパラメータ: 0.norm1.weight
更新されるパラメータ: 0.norm1.bias
更新されるパラメータ: 0.norm2.weight
更新されるパラメータ: 0.norm2.bias
更新されるパラメータ: 1.self_attn.in_proj_weight
更新されるパラメータ: 1.self_attn.in_proj_bias
更新されるパラメータ: 1.self_attn.out_proj.weight
更新されるパラメータ: 1.self_attn.out_proj.bias
更新されるパラメータ: 1.linear1.weight
更新されるパラメータ: 1.linear1.bias
更新されるパラメータ: 1.linear2.weight
更新されるパラメータ: 1.linear2.bias
更新されるパラメータ: 1.norm1.weight
更新されるパラメータ: 1.norm1.bias
更新されるパラメータ: 1.norm2.weight
更新されるパラメータ: 1.norm2.bias
更新されるパラメータ: weight
更新されるパラメータ: bias


In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params_to_update, lr=config.learning_rate)

### step数で管理してみた

In [18]:
# step数で管理するためのデータローダー関数
def infinite_loader(dataloader):
    while True:
        for batch in dataloader:
            yield batch

data_iter = infinite_loader(train_loader)  # epochではなく、step数で計測

In [19]:
from tqdm import tqdm
max_iters = 2500 # step数 (更新回数) = epoch数 (LOOPの回数)× ミニバッチ分割数

In [20]:
pbar = tqdm(range(max_iters))
model.train()                            # trainモードを明示
for step in pbar:
    batch = next(data_iter)
    x = batch["input_ids"].to(device)
    t = batch["labels"].to(device)
    optimizer.zero_grad()
    y = model(x)
    loss = criterion(y,t)
    acc = accuracy(y, t)    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.max_grad_norm)  # 勾配クリップ
    optimizer.step()
    # set_postfixは辞書が引数になる
    pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc":f"{acc:.3f}"})
    if (step+1)%500 == 0:
        #lr_now = scheduler.get_last_lr()[0]
        tqdm.write(f"{step+1}-step:\tloss:{loss.item():.3f}\tacc:{acc:.3f}")# \tlr:{lr_now:.5f}")


 21%|██        | 518/2500 [00:05<00:20, 95.47it/s, loss=1.2578, acc=0.875]

500-step:	loss:1.304	acc:0.750


 41%|████      | 1014/2500 [00:11<00:14, 105.33it/s, loss=0.9839, acc=0.875]

1000-step:	loss:1.211	acc:0.625


 61%|██████    | 1514/2500 [00:16<00:09, 104.32it/s, loss=0.4957, acc=1.000]

1500-step:	loss:0.682	acc:0.750


 80%|████████  | 2010/2500 [00:21<00:05, 91.91it/s, loss=0.2896, acc=1.000] 

2000-step:	loss:0.396	acc:0.875


100%|██████████| 2500/2500 [00:26<00:00, 95.15it/s, loss=0.6292, acc=0.750] 

2500-step:	loss:0.629	acc:0.750


## 検証
* test_dataにたいしてもSimpleDatasetとDataLoaderを利用する
* 面倒なので全部を1つのバッチとして扱ってみた😆

In [21]:
# (1) テストデータ全件を 1バッチ で処理する DataLoader を作成
test_dataset = SimpleDataset(test_data)
test_all_loader = DataLoader(
    test_dataset, 
    batch_size=len(test_dataset),  # 全件を一つのバッチにする
    shuffle=False, 
    collate_fn=hf_collator
)

# (2) 1回だけループを回してデータを取り出す
# next(iter(...)): よく忘れるのでメモ 
test_batch = next(iter(test_all_loader))

# (3) 検証データで推論
model.eval()
with torch.inference_mode():
    x_test = test_batch["input_ids"].to(device)
    t_test = test_batch["labels"].to(device)

    y_test = model(x_test)
    acc = accuracy(y_test, t_test)

print(f"検証精度: {acc}")

検証精度: 0.812
